# SeisBench dataset construction

Combines consensus-labelled waveform windows with noise examples and writes the local dataset and metadata used by the training notebooks.


## Dataset design

Keep full QC-passed windows, reject implausible S–P intervals, and assign splits by event before generating synthetic noise from eligible pre-P segments. Training notebooks crop and normalize inputs for each architecture.

In [1]:
# ==========================================
# 1. IMPORTS AND PATHS
# ==========================================
import os
import sys

# On Windows the Jupyter kernel starts without conda activation: add the DLL
# directories to PATH so torch/seisbench can load (avoids WinError 127).
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
for _dll_dir in (os.path.join(sys.prefix, "Library", "bin"),
                 os.path.join(sys.prefix, "Library", "mingw-w64", "bin"),
                 sys.prefix):
    if os.path.isdir(_dll_dir):
        os.environ["PATH"] = _dll_dir + os.pathsep + os.environ.get("PATH", "")

import torch  # noqa: E402  (import before obspy/seisbench on Windows)

import logging
logging.getLogger("seisbench").setLevel(logging.ERROR)

import glob
import importlib
from pathlib import Path

import numpy as np
import pandas as pd
import seisbench.data as sbd


def _find_repo_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "data" / "raw").is_dir() and (p / "src").is_dir():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(REPO_ROOT / "src" / "preprocessing"))
sys.path.insert(0, str(REPO_ROOT / "src" / "labeling"))
import preprocessing_utils as pp   # noqa: E402  (same conditioning as the labelling stage)
import consensus_utils as cu       # noqa: E402  (synth_noise)
importlib.reload(pp)
importlib.reload(cu)

WAVEFORMS_DIR = REPO_ROOT / "data" / "raw" / "waveforms"
LABELING_DIR = REPO_ROOT / "data" / "processed" / "labeling"
DATASET_DIR = REPO_ROOT / "data" / "processed" / "dataset"
DATASET_DIR.mkdir(parents=True, exist_ok=True)
METADATA_PATH = DATASET_DIR / "metadata.csv"
WAVEFORMS_PATH = DATASET_DIR / "waveforms.hdf5"

print(f"Repo root: {REPO_ROOT}")
print(f"Labels in: {LABELING_DIR}")
print(f"Dataset -> {DATASET_DIR}")

Repo root: M:\Github\TFM_SeismicPhasePicker_ColombianAndes
Labels in: M:\Github\TFM_SeismicPhasePicker_ColombianAndes\data\processed\labeling
Dataset -> M:\Github\TFM_SeismicPhasePicker_ColombianAndes\data\processed\dataset


In [2]:
# ==========================================
# 2. CONFIGURATION
# ==========================================
# Select stations with saved labels, or specify a subset.
STATIONS = None   # None = ALL stations with labels_<STA>.csv

# Event mode keeps records with the same event key in one split.
# Random mode assigns individual windows within each category.
SPLIT_MODE = "event"
SPLIT_FRACTIONS = (0.70, 0.15, 0.15)   # train / dev / test
SEED = 42

# Noise bank
INCLUDE_SYNTH_NOISE = True
NOISE_TO_EQ_RATIO = 1.0    # retained for configuration compatibility; balance is exact
SYNTH_PRE_A = 200                 # source segment start [2 s] at 100 Hz
SYNTH_PRE_B = 2700                # source segment end by default [27 s] at 100 Hz
SYNTH_SAFETY_MARGIN_S = 2.0       # cut this many seconds before the real P when inside the segment
SYNTH_MIN_SEGMENT_S = 2.0         # minimum usable pre-event segment after the cut (s)
SP_MIN_S = 1.0                    # minimum admissible S-P separation (s)
SP_MAX_S = 60.0                   # maximum admissible S-P separation (s)

STORE_NPTS = pp.NPTS       # 15001 (150 s); cropped per model at training time
PRINT_EVERY = 200

print(f"STATIONS = {STATIONS if STATIONS else 'all'}")
print(f"split = {SPLIT_MODE} {SPLIT_FRACTIONS} (seed {SEED}) | synthetic noise: {INCLUDE_SYNTH_NOISE} "
      f"(synthetic balance = exact per split; pre-P segment cut {SYNTH_SAFETY_MARGIN_S} s before P)")

STATIONS = all
split = event (0.7, 0.15, 0.15) (seed 42) | synthetic noise: True (synthetic balance = exact per split; pre-P segment cut 2.0 s before P)


In [3]:
# ==========================================
# 3. LOAD LABELS AND ASSIGN THE SPLIT
# ==========================================
def _load_labels(stations):
    files = sorted(glob.glob(str(LABELING_DIR / "labels_*.csv")))
    frames = []
    for f in files:
        sta = os.path.basename(f)[len("labels_"):-len(".csv")]
        if stations and sta not in stations:
            continue
        frames.append(pd.read_csv(f, on_bad_lines="skip"))
    if not frames:
        raise FileNotFoundError("No labels_<STA>.csv found for the requested stations.")
    return pd.concat(frames, ignore_index=True)

labels = _load_labels(STATIONS)
labels = labels[labels["category"].isin(["earthquake", "noise"])].reset_index(drop=True)
n_selected = len(labels)
n_eq_selected = int((labels.category == "earthquake").sum())
n_noise_selected = int((labels.category == "noise").sum())
print(f"selected: {n_selected} (earthquake={n_eq_selected}, noise={n_noise_selected})")

# Check S–P separation only where both pseudo-arrivals are present.
sp = labels["s_time_s"] - labels["p_time_s"]
has_both = (labels["category"] == "earthquake") & labels["p_time_s"].notna() & labels["s_time_s"].notna()
bad_lt = has_both & (sp < SP_MIN_S)
bad_gt = has_both & (sp > SP_MAX_S)
bad_sp = bad_lt | bad_gt
labels = labels.loc[~bad_sp].reset_index(drop=True)
print(f"S-P depuration: removed {int(bad_sp.sum())} = {int(bad_lt.sum())} below {SP_MIN_S} s + {int(bad_gt.sum())} above {SP_MAX_S} s")
print(f"after depuration: {len(labels)} (earthquake={(labels.category == 'earthquake').sum()}, noise={(labels.category == 'noise').sum()})")

rng = np.random.default_rng(SEED)

def _assign_split(n):
    """train/dev/test labels for n rows, in SPLIT_FRACTIONS proportions."""
    idx = rng.permutation(n)
    n_tr = int(round(SPLIT_FRACTIONS[0] * n))
    n_dev = int(round(SPLIT_FRACTIONS[1] * n))
    out = np.empty(n, dtype=object)
    out[idx[:n_tr]] = "train"
    out[idx[n_tr:n_tr + n_dev]] = "dev"
    out[idx[n_tr + n_dev:]] = "test"
    return out

# Assign the split according to SPLIT_MODE
if SPLIT_MODE == "event":
    # Approximate event keys from rounded location, depth, and magnitude.
    ev = (labels[["source_latitude", "source_longitude", "source_depth_km",
                  "source_magnitude"]].round(4).astype(str).agg("|".join, axis=1))
    keys = ev.unique()
    key_to_split = dict(zip(keys, _assign_split(len(keys))))
    labels["split"] = ev.map(key_to_split)
    print(f"split by EVENT: {len(keys)} unique events")
elif SPLIT_MODE == "random":
    # per-window random, stratified by class (OBS style)
    labels["split"] = ""
    for cat, g in labels.groupby("category"):
        labels.loc[g.index, "split"] = _assign_split(len(g))
    print("RANDOM per-window split (stratified by class)")
else:
    raise ValueError("SPLIT_MODE must be 'event' or 'random'")

n_eq = int((labels.category == "earthquake").sum())
n_noise_real = int((labels.category == "noise").sum())
n_synth = max(0, n_eq - n_noise_real) if INCLUDE_SYNTH_NOISE else 0
print(f"earthquake: {n_eq}  |  real noise: {n_noise_real}  |  synthetic noise to generate: {n_synth}")

# Select synthetic-noise sources within each split. The pre-P segment must
# retain the safety margin and minimum duration configured above.
eq_dep = labels[labels.category == "earthquake"]
src_pool = eq_dep[eq_dep["p_time_s"].notna()].copy()
src_pool["pre_b_sample"] = np.minimum(
    SYNTH_PRE_B, (src_pool["p_time_s"] - SYNTH_SAFETY_MARGIN_S) * 100
).astype(int)
src_pool = src_pool[
    src_pool["pre_b_sample"] - SYNTH_PRE_A >= SYNTH_MIN_SEGMENT_S * 100
].copy()
n_p_short = int(
    (eq_dep["p_time_s"].notna()
     & ((eq_dep["p_time_s"] - SYNTH_SAFETY_MARGIN_S) * 100 - SYNTH_PRE_A
        < SYNTH_MIN_SEGMENT_S * 100)).sum()
)
n_p_unknown = int(eq_dep["p_time_s"].isna().sum())
print(f"source pool: {len(src_pool)} with usable pre-P segment")
print(f"source exclusions: {n_p_short} with P too early for a safe cut; {n_p_unknown} without P label")

eq_by_split = labels[labels.category == "earthquake"].groupby("split").size()
noise_by_split = labels[labels.category == "noise"].groupby("split").size()
need_by_split = eq_by_split.sub(noise_by_split, fill_value=0).astype(int)
pool_by_split = src_pool.groupby("split").size()
print("counts by category and split:")
print(labels.groupby(["category", "split"]).size().unstack(fill_value=0).to_string())
print("synthetic need by split:")
for _split in ["train", "dev", "test"]:
    print(f"  {_split}: need={int(need_by_split.get(_split, 0))}, pool={int(pool_by_split.get(_split, 0))}")
if STATIONS is None:
    assert (n_selected, n_eq_selected, n_noise_selected) == (66498, 53861, 12637)
    assert (int(bad_sp.sum()), n_eq, n_noise_real, n_synth, len(src_pool)) == (571, 53290, 12637, 40653, 49318)
    assert (int(n_p_short), int(n_p_unknown)) == (21, 3951)
    assert len(labels) + n_synth == 106580
    assert all(int(pool_by_split.get(s, 0)) >= int(need_by_split.get(s, 0)) for s in ["train", "dev", "test"])

selected: 66498 (earthquake=53861, noise=12637)
S-P depuration: removed 571 = 265 below 1.0 s + 306 above 60.0 s
after depuration: 65927 (earthquake=53290, noise=12637)
split by EVENT: 19704 unique events
earthquake: 53290  |  real noise: 12637  |  synthetic noise to generate: 40653
source pool: 49318 with usable pre-P segment
source exclusions: 21 with P too early for a safe cut; 3951 without P label
counts by category and split:
split        dev  test  train
category                     
earthquake  8207  7783  37300
noise       1923  1916   8798
synthetic need by split:
  train: need=28502, pool=34565
  dev: need=6284, pool=7575
  test: need=5867, pool=7178


In [4]:
# ==========================================
# 4. METADATA HELPERS
# ==========================================
def _f(v):
    """Safe float (NaN if empty)."""
    try:
        return float(v)
    except (TypeError, ValueError):
        return np.nan

def base_meta(r):
    """Common SeisBench columns built from one label row."""
    return {
        "source_id": str(r["timestamp"]),
        "source_latitude_deg": _f(r.get("source_latitude")),
        "source_longitude_deg": _f(r.get("source_longitude")),
        "source_depth_km": _f(r.get("source_depth_km")),
        "source_magnitude": _f(r.get("source_magnitude")),
        "station_code": str(r["station_code"]),
        "station_latitude_deg": _f(r.get("station_latitude")),
        "station_longitude_deg": _f(r.get("station_longitude")),
        "station_elevation_m": _f(r.get("station_elevation_m")),
        "path_ep_distance_km": _f(r.get("distance_km")),
        "path_back_azimuth_deg": _f(r.get("back_azimuth")),
        "trace_channel": "HH",
        "trace_sampling_rate_hz": pp.SAMPLING_RATE_HZ,
        # pseudo-label traceability
        "trace_p_n_models": r.get("p_n_models"),
        "trace_s_n_models": r.get("s_n_models"),
        "trace_p_prob": _f(r.get("p_prob")),
        "trace_s_prob": _f(r.get("s_prob")),
        "trace_snr_p_db": _f(r.get("snr_p_db")),
        "trace_snr_s_db": _f(r.get("snr_s_db")),
    }

def eq_meta(r):
    m = base_meta(r)
    m["trace_name"] = str(r["trace_name"])
    m["trace_category"] = "earthquake"
    m["split"] = r["split"]
    if not pd.isna(r.get("p_sample")):
        m["trace_p_arrival_sample"] = int(r["p_sample"])
        m["trace_p_status"] = "automatic"
    if not pd.isna(r.get("s_sample")):
        m["trace_s_arrival_sample"] = int(r["s_sample"])
        m["trace_s_status"] = "automatic"
    return m

def noise_meta(r):
    m = base_meta(r)
    m["trace_name"] = str(r["trace_name"])
    m["trace_category"] = "noise"
    m["split"] = r["split"]
    return m

print("helpers ready")

helpers ready


In [5]:
# ==========================================
# 5. WRITE THE SEISBENCH DATASET
# ==========================================
# Write accepted windows using the shared preprocessing, then add surrogate noise.
def _load_wave(station, timestamp):
    arr = pp.load_window_array(WAVEFORMS_DIR / station, timestamp,
                               station_code=station, preprocess=True)
    if arr is None:
        return None
    return np.ascontiguousarray(arr[:, :STORE_NPTS].astype(np.float32))

eq_rows = labels[labels.category == "earthquake"]
noise_rows = labels[labels.category == "noise"]
total = len(labels) + n_synth

written = {"earthquake": 0, "noise": 0, "noise_synth": 0}
skipped = 0

with sbd.WaveformDataWriter(METADATA_PATH, WAVEFORMS_PATH) as writer:
    writer.data_format = {
        "dimension_order": "CW",
        "component_order": "ZNE",
        "sampling_rate": pp.SAMPLING_RATE_HZ,
        "measurement": "velocity",
        "unit": "counts",
        "instrument_response": "not restituted",
    }
    writer.set_total(total)

    # --- earthquakes ---
    for i, (_, r) in enumerate(eq_rows.iterrows()):
        data = _load_wave(r["station_code"], r["timestamp"])
        if data is None:
            skipped += 1
            continue
        writer.add_trace(eq_meta(r), data)
        written["earthquake"] += 1
        if (i + 1) % PRINT_EVERY == 0:
            print(f"  earthquake {i + 1}/{len(eq_rows)} (skip {skipped})")

    # --- real noise ---
    for i, (_, r) in enumerate(noise_rows.iterrows()):
        data = _load_wave(r["station_code"], r["timestamp"])
        if data is None:
            skipped += 1
            continue
        writer.add_trace(noise_meta(r), data)
        written["noise"] += 1
        if (i + 1) % PRINT_EVERY == 0:
            print(f"  noise {i + 1}/{len(noise_rows)} (skip {skipped})")

    # --- synthetic noise (prolongation of pre-event segments) ---
    if n_synth > 0:
        # Sample source events within their assigned split.
        src_parts = []
        for _split in ["train", "dev", "test"]:
            need = int(need_by_split.get(_split, 0))
            pool_s = src_pool[src_pool["split"] == _split]
            if len(pool_s) < need:
                raise ValueError(f"Insufficient source pool for {_split}: {len(pool_s)} < {need}")
            if need:
                src_parts.append(pool_s.sample(n=need, replace=False, random_state=SEED))
        src = pd.concat(src_parts, ignore_index=True)
        if len(src) != n_synth:
            raise AssertionError(f"Sampled {len(src)} synthetic sources, expected {n_synth}")
        for i, (_, r) in enumerate(src.iterrows()):
            base = _load_wave(r["station_code"], r["timestamp"])
            if base is None:
                skipped += 1
                continue
            noise = cu.synth_noise(base, STORE_NPTS, pre_a=SYNTH_PRE_A, pre_b=int(r["pre_b_sample"]),
                                   rng=np.random.default_rng(SEED + i))
            m = base_meta(r)
            m["trace_name"] = f"synthnoise_{i:06d}_{r['station_code']}"
            m["source_id"] = f"synth_{i:06d}"
            m["trace_category"] = "noise"
            m["trace_synthetic"] = True
            m["split"] = r["split"]  # inherits the split of the source event
            # synthetic noise carries no picks or source magnitude
            for k in ("source_magnitude", "source_depth_km", "path_ep_distance_km"):
                m[k] = np.nan
            writer.add_trace(m, np.ascontiguousarray(noise.astype(np.float32)))
            written["noise_synth"] += 1
            if (i + 1) % PRINT_EVERY == 0:
                print(f"  synth-noise {i + 1}/{n_synth}")

print("\nDone.")
print(f"  earthquakes written : {written['earthquake']}")
print(f"  real noise written  : {written['noise']}")
print(f"  synthetic noise     : {written['noise_synth']}")
print(f"  skipped (QC/None)   : {skipped}")
print(f"  metadata -> {METADATA_PATH}")
print(f"  waveforms -> {WAVEFORMS_PATH}")

Traces converted:   0%|          | 203/106580 [00:10<1:36:50, 18.31it/s]

  earthquake 200/53290 (skip 0)


Traces converted:   0%|          | 402/106580 [00:19<1:15:53, 23.32it/s]

  earthquake 400/53290 (skip 0)


Traces converted:   1%|          | 605/106580 [00:28<1:18:12, 22.58it/s]

...

  synth-noise 40400/40653


Traces converted: 100%|█████████▉| 106528/106580 [3:10:29<00:06,  8.55it/s]

  synth-noise 40600/40653


Traces converted: 100%|██████████| 106580/106580 [3:10:47<00:00,  9.31it/s]


Done.
  earthquakes written : 53290
  real noise written  : 12637
  synthetic noise     : 40653
  skipped (QC/None)   : 0
  metadata -> M:\Github\TFM_SeismicPhasePicker_ColombianAndes\data\processed\dataset\metadata.csv
  waveforms -> M:\Github\TFM_SeismicPhasePicker_ColombianAndes\data\processed\dataset\waveforms.hdf5


In [6]:
# ==========================================
# 6. QUICK DATASET CHECK
# ==========================================
meta = pd.read_csv(METADATA_PATH, low_memory=False)
print(f"Total traces: {len(meta)}\n")
print("By category and split:")
by_split = meta.groupby(["trace_category", "split"]).size().unstack(fill_value=0)
print(by_split.to_string())
for _split in ["train", "dev", "test"]:
    eq_count = int(by_split.get(_split, pd.Series(dtype=int)).get("earthquake", 0))
    noise_count = int(by_split.get(_split, pd.Series(dtype=int)).get("noise", 0))
    if eq_count != noise_count:
        raise AssertionError(f"Unbalanced split {_split}: earthquake={eq_count}, noise={noise_count}")
print("Exact class balance verified in train/dev/test.")

eqm = meta[meta.trace_category == "earthquake"]
if len(eqm):
    has_p = eqm["trace_p_arrival_sample"].notna().mean() * 100 if "trace_p_arrival_sample" in eqm else 0
    has_s = eqm["trace_s_arrival_sample"].notna().mean() * 100 if "trace_s_arrival_sample" in eqm else 0
    print(f"\nearthquake with P: {has_p:.0f}%  |  with S: {has_s:.0f}%")

# Load with the SeisBench API to confirm it opens correctly
data = sbd.WaveformDataset(DATASET_DIR, sampling_rate=pp.SAMPLING_RATE_HZ)
print(f"\nSeisBench WaveformDataset OK: {len(data)} traces")
if len(data) != 106580:
    raise AssertionError(f"Expected 106580 traces, found {len(data)}")
print("splits:", {s: len(data.get_split(s)) for s in ["train", "dev", "test"]})

Total traces: 106580

By category and split:
split            dev  test  train
trace_category                   
earthquake      8207  7783  37300
noise           8207  7783  37300
Exact class balance verified in train/dev/test.

earthquake with P: 93%  |  with S: 69%

SeisBench WaveformDataset OK: 106580 traces
splits: {'train': 74600, 'dev': 16414, 'test': 15566}


In [7]:
# ==========================================
# 7. FINAL DATASET CHARACTERIZATION
# ==========================================
meta = pd.read_csv(METADATA_PATH, low_memory=False)
meta["is_synth"] = meta["trace_synthetic"].astype(str).str.strip().str.lower().isin(["true", "1"])
eqm = meta[meta.trace_category == "earthquake"]

# 1) Phase-label coverage by split
print("Phase-label coverage (earthquakes):")
print("  overall P =", f"{eqm['trace_p_arrival_sample'].notna().mean()*100:.1f}%",
      "| S =", f"{eqm['trace_s_arrival_sample'].notna().mean()*100:.1f}%")
cov = eqm.groupby("split")[["trace_p_arrival_sample", "trace_s_arrival_sample"]].apply(
    lambda g: (g.notna().mean() * 100).round(1))
print(cov.to_string())

# 2) Consensus strength of retained P labels
print("\nConsensus strength (P n_models, fraction by split):")
print(eqm.groupby("split")["trace_p_n_models"].value_counts(normalize=True).round(3).to_string())

# 3) Per-station composition
comp = pd.DataFrame({
    "eq": eqm.groupby("station_code").size(),
    "noise_real": meta[(meta.trace_category == "noise") & ~meta.is_synth].groupby("station_code").size(),
    "noise_synth": meta[meta.is_synth].groupby("station_code").size(),
}).fillna(0).astype(int)
print("\nPer-station composition:")
print(comp.to_string())

# 4) Event-key leakage across splits (real windows only)
for c in ["source_latitude_deg", "source_longitude_deg", "source_depth_km", "source_magnitude"]:
    meta[c] = pd.to_numeric(meta[c], errors="coerce")
real = meta[~meta.is_synth]
key = real[["source_latitude_deg", "source_longitude_deg", "source_depth_km", "source_magnitude"]].apply(
    lambda r: "|".join(str(round(float(v), 4)) for v in r), axis=1)
n_splits = pd.DataFrame({"key": key, "split": real["split"]}).groupby("key")["split"].nunique()
print(f"\nEvent-key leakage check: {len(n_splits)} unique events, "
      f"{(n_splits > 1).sum()} spanning >1 split")

# 5) Synthetic source uniqueness
syn = meta[meta.is_synth]
print(f"Synthetic traces: {len(syn)}, unique source_id: {syn['source_id'].nunique()}, "
      f"source stations: {syn['station_code'].nunique()}")

# 6) Earthquake characterization (median / P10 / P90)
print("\nEarthquake characterization (median / P10 / P90):")
for col in ["source_magnitude", "path_ep_distance_km", "trace_snr_p_db", "trace_snr_s_db"]:
    v = pd.to_numeric(eqm[col], errors="coerce").dropna()
    print(f"  {col}: n={len(v)} mediana={v.median():.2f} P10={v.quantile(0.1):.2f} P90={v.quantile(0.9):.2f}")

# 7) Duplicates
print(f"\nDuplicate trace_name: {meta['trace_name'].duplicated().sum()}")


Phase-label coverage (earthquakes):
  overall P = 92.6% | S = 68.5%
       trace_p_arrival_sample  trace_s_arrival_sample
split                                                
dev                      92.3                    69.2
test                     92.2                    67.8
train                    92.7                    68.5

Consensus strength (P n_models, fraction by split):
split  trace_p_n_models
dev    3                   0.673
       4                   0.251
       0                   0.077
test   3                   0.675
       4                   0.247
       0                   0.078
train  3                   0.678
       4                   0.249
       0                   0.073

Per-station composition:
                eq  noise_real  noise_synth
station_code                               
BRJC          1561         961         1287
CBOC          5572         419         4428
CHI           2971         913         2282
DBB           1639         167         136

In [8]:
# ==========================================
# 8. PHASE-COVERAGE BREAKDOWN BY SPLIT
# ==========================================
meta = pd.read_csv(METADATA_PATH, low_memory=False)
eqm = meta[meta.trace_category == "earthquake"].copy()
has_p = eqm["trace_p_arrival_sample"].notna()
has_s = eqm["trace_s_arrival_sample"].notna()
eqm["fases"] = "solo_S"
eqm.loc[has_p & ~has_s, "fases"] = "solo_P"
eqm.loc[has_p & has_s, "fases"] = "ambas"
print("Earthquake windows by retained phases and split:")
print(eqm.groupby(["split", "fases"]).size().unstack(fill_value=0).to_string())
print("\nTotals: ambas =", int((has_p & has_s).sum()),
      "| solo_P =", int((has_p & ~has_s).sum()),
      "| solo_S =", int((~has_p & has_s).sum()))


Earthquake windows by retained phases and split:
fases  ambas  solo_P  solo_S
split                       
dev     5048    2531     628
test    4674    2505     604
train  22831   11750    2719

Totals: ambas = 32553 | solo_P = 16786 | solo_S = 3951
